# Basic Fitting Workflow

This notebook demonstrates the standard fitting workflow for time- and energy-resolved spectroscopy data in five steps. The core is three models that build on each other:

1. **Select the data**: Load and inspect the data, then set the fit limits, i.e. energy/time region to fit.
2. **Fit the baseline** (`base`): Select the time slices making up the ground-state spectrum. Fit the well-determined line-shape parameters that are time-independent so the following fits are more robust and faster.
3. **Slice-by-slice fitting** (`SbS`): Fit each time slice independently, with the peak position being the only free fit parameter while the line shape is pinned to the baseline result. This reveals how the position moves in time without assuming a functional form.
4. **Global 2D fitting** (`2D`): Perform one global fit over the whole dataset, where the peak position follows a parametric law in time defined in `models_time.yaml`. This approach drastically reduces the overall number of free fit parameters compared to fitting a parametric law in time to the SbS fit results.
5. **Inspect the results**: Because the data is synthetic we can compare the fit results to the known parameters used to generate the data (`data/models_energy_truth.yaml`, `data/models_time_truth.yaml`).

In [ ]:
from pathlib import Path

import numpy as np
import trspecfit

## 1. Load and Inspect Data

### 1.1 Load Data

Load simulated time- and energy-resolved spectroscopy data from CSV files. The data is synthetic, generated by [`data/generate_data.ipynb`](data/generate_data.ipynb) which can be re-run to regenerate or change the data.
<br><br>
The 2D dataset is a stack of 1D energy spectra, one per time delay. The `data` array is `[time, energy]`, i.e. rows are time, columns are energy.

In [ ]:
# Create project
project = trspecfit.Project(path=Path.cwd())

# Directory containing data files
data_folder = 'data'

# Add file to project
file = trspecfit.File(
    parent_project=project,
    path=data_folder,
    data=np.loadtxt(project.path / data_folder / 'data.csv', delimiter=','),
    energy=np.loadtxt(project.path / data_folder / 'energy.csv'),
    time=np.loadtxt(project.path / data_folder / 'time.csv')
)

### 1.2 Inspect Data

A Gaussian–Lorentzian-product (`GLP`) peak sits on a linear background (`LinBack`). After t = 0 the peak position shifts in time, blurred by a Gaussian instrument response (IRF).

In [ ]:
# Visualize full dataset
file.describe()

### 1.3 Define Fitting Region

Set energy and time limits for the fits. They appear as black, dotted lines in the plot.

In [ ]:
# Set fitting limits (absolute values)
file.set_fit_limits(
    energy_limits=[5, 18],  # Energy range
    time_limits=[-20, 100]  # Time range
)

## 2. Fit Baseline / Ground State Spectrum

### 2.1 Define Baseline Spectrum

Extract the ground-state spectrum by averaging the pre-perturbation time slices. Make sure that the instrument response function is not bleeding into the baseline by lowering `time_stop` and re-fitting the baseline.

In [ ]:
# Define baseline using absolute time values
file.define_baseline(
    time_start=-20,
    time_stop=-10,
    time_type='abs'  # Use absolute time values (or 'ind' for indices)
)

### 2.2 Fit Baseline Spectrum

Fit the ground state spectrum leaving most fit parameters free to vary. The model is defined in `models_energy.yaml`. YAML model files are passed by name and resolved relative to the project directory passed to `Project(...)` in §1.1.
<br><br>
 During time-dependent fits, the values of all fixed parameters (`vary=False`) will be set to the baseline fit result values.

In [ ]:
# Load baseline model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='base'
)

# Inspect model structure and parameters
file.describe_model(model_info='base', detail=0)
#file.model_active.visualize() # visualize model as a graph

In [ ]:
# Fit baseline model to data
# stages=1: single optimization
# stages=2: two-stage (global + local)
file.fit_baseline(
    model_name='base',
    stages=2,
    try_ci=0  # Confidence intervals: see the fit-options table below and 12_uncertainty_mcmc
)

## 3. Slice-by-Slice Fitting

Fit each time slice independently to see how the parameters evolve over time, without assuming the form of that evolution. In this example, the peak position `GLP_01_x0` is the only parameter free to vary as a function of time.

In [ ]:
# Load Slice-by-Slice model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='SbS'
)

file.describe_model('SbS', detail=0)

In [ ]:
# Perform Slice-by-Slice fit
file.fit_slice_by_slice(
    model_name='SbS',
    stages=1,
    try_ci=0  # Minimizer needs >=2 varying parameters to determine confidence intervals
)

## 4. Global 2D Fitting

The slice-by-slice fit showed `GLP_01_x0` rising after t = 0 and leveling off — an exponential approach. So instead of leaving the position free in every slice, we model it with a single law in time: an exponential shift (`expFun`) convolved with a Gaussian instrument response (`gaussCONV`), i.e. the `MonoExpIRF` model in `models_time.yaml`.
<br><br>
**Why global?**
Fitting all time points together constrains the dynamics within the entire dataset, reducing overfitting and giving more robust, physically meaningful parameters than the independent per-slice fits. While one could fit the SbS results, i.e. `GLP_01_x0(t)` with the `MonoExpIRF` model, the overall number of free fit parameters in this approach would be hundreds more than in the global fit shown here.

In [ ]:
# Load 2D model
file.load_model(
    model_yaml='models_energy.yaml',
    model_info='2D'
)

file.describe_model(model_info='2D', detail=0)

In [ ]:
# Add time dependence to parameter
file.add_time_dependence(
    target_model='2D',
    target_parameter='GLP_01_x0',
    dynamics_yaml='models_time.yaml',
    dynamics_model='MonoExpIRF',
)

print("\n=== Model with Time Dependence ===")
file.describe_model(model_info='2D', detail=1)
#file.model_active.visualize() # visualize model as a graph

In [ ]:
# Perform global 2D fit
file.fit_2d(
    model_name='2D',
    stages=2,
    try_ci=0  # Skip confidence interval estimation for a faster fit
)

## 5. Inspect the Fit Results

`file.get_fit_results(fit_type=...)` returns a `pandas.DataFrame` with one row per parameter for `baseline` / `2d` or one row per time slice for `sbs`.<br>
For more details on accessing fit results and metadata, see [`10_model_comparison`](../10_model_comparison/example.ipynb).

Because the data is synthetic, we can compare the fit results to the known truth printed at the end of `data/generate_data.ipynb`: the baseline fit recovers the peak shape (position `x0 ≈ 9`, amplitude `≈ 12`, and width `≈ 1`) and linear background, and the global 2D fit recovers the dynamics (time constant `tau ≈ 50`, IRF width `SD ≈ 3`, shift amplitude `≈ 5`).

In [ ]:
file.get_fit_results(fit_type='2d')
# if stages=1, `init_value` is the initial guess from the model definition
# if stages=2, `init_value` is the result of the stage 1 fit

## Tips for Fitting

**Fitting Strategy**
- Always fit baseline first to establish good initial values
- Use Slice-by-Slice to identify which parameters are time-dependent
- Apply time-dependence only to parameters that clearly evolve
- Start with simple time-dependencies before complex ones

**Fit Levels**
- `stages=1`: Use a single optimization pass with e.g. least squares algorithm if your initial guess is close to the ground truth.
- `stages=2`: Use a global fit as the first algorithm to avoid getting stuck in local minima. Use a local fit algorithm as a second stage, e.g. least squares, for more robust results.

**Common Fit Options**

The `fit_*` methods expose `stages` directly and pass most other fit-control keywords through to `trspecfit.fitlib.fit_wrapper`.

| Option | Applies to | Default | Use when |
|--------|------------|---------|----------|
| `stages` | baseline, SbS, 2D | `1` | `1` runs `fit_alg_1`; `2` runs `fit_alg_1` followed by `fit_alg_2`. |
| `fit_alg_1` | baseline, SbS, 2D | `'Nelder'` | Robust first pass. Use `'leastsq'` for fast local fits when the initial guess is already close. |
| `fit_alg_2` | baseline, SbS, 2D | `'leastsq'` | Local refinement used only when `stages=2`. |
| `try_ci` | baseline, SbS, 2D | `1` | Set `0` for fast exploratory fits; set `1` for lmfit confidence intervals. |
| `ci_sigmas` | baseline, SbS, 2D | `[1, 2, 3]` | Confidence levels (in σ units) reported when `try_ci=1`. |
| `mc_settings` | baseline, SbS, 2D | off | Run MCMC uncertainty estimation; see `12_uncertainty_mcmc`. |
| `n_workers` | SbS only | `None` | Control parallel Slice-by-Slice fitting; `None` auto-selects workers, `1` runs serially. |
| `seed_source` / `seed_adapt` | SbS only | `'baseline'` / `'argmax_shift'` | Control how each independent time slice is initialized. |

## Next Steps

**Uncertainty Estimation**
- `try_ci=1`: Use `lmfit.conf_interval` for confidence intervals
- For robust MCMC-based uncertainty estimation, see [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb)

**Compare, Save & Export Fit Results**
- For comparing two models on the same data, see [`10_model_comparison`](../10_model_comparison/example.ipynb)
- To save, load, and export your fits — CSV/PNG for Origin / MATLAB, or round-trippable HDF5 archives, see [`11_save_load_export`](../11_save_load_export/example.ipynb)

**Explore More Complex Models**
- For fitting with parameter expressions, see [`02_dependent_parameters`](../02_dependent_parameters/example.ipynb) 
- For fitting datasets with multiple sub-cycles, see [`03_multi_cycle_dynamics`](../03_multi_cycle_dynamics/example.ipynb)
- For fit parameters that vary along an auxiliary axis, see [`04_parameter_profiles`](../04_parameter_profiles/example.ipynb)